# Project 4: Predicting Titanic Fare Using Regression

**Student:** Sandra Otubushin

**Course:** Machine Learning

**Date:** July 2026

## Project Overview

In this project, I use regression models to predict the fare paid by Titanic passengers. I compare multiple regression techniques, evaluate model performance using RMSE, MAE, and R², and analyze how different features affect prediction accuracy.

In [1]:
import numpy as np
import seaborn as sns
from sklearn.linear_model import (
    ElasticNet,
    LinearRegression,
    Ridge,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

## Section 1: Import and Inspect the Data

In [2]:
# Load Titanic dataset

titanic = sns.load_dataset("titanic")

# Display first five rows
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [3]:
# Display dataset information

titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [4]:
# Summary statistics

titanic.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


## Section 2: Data Exploration and Preparation

In [5]:
# Fill missing values

titanic["age"] = titanic["age"].fillna(titanic["age"].median())

# Remove rows missing fare

titanic = titanic.dropna(subset=["fare"])

# Create family size

titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1

# Convert sex to numeric

titanic["sex_num"] = titanic["sex"].map({"male": 0, "female": 1})

# Verify

titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,family_size,sex_num
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,2,0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,2,1
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,1,1
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,2,1
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,1,0


## Section 3: Feature Selection and Justification

In [6]:
# Case 1: Age only
X1 = titanic[["age"]]
y1 = titanic["fare"]

# Case 2: Family Size only
X2 = titanic[["family_size"]]
y2 = titanic["fare"]

# Case 3: Age and Family Size
X3 = titanic[["age", "family_size"]]
y3 = titanic["fare"]

# Case 4: Age, Family Size, and Sex
X4 = titanic[["age", "family_size", "sex_num"]]
y4 = titanic["fare"]

### Reflection

**Why might these features affect a passenger's fare?**

Passenger age, family size, and sex may all influence the fare because families often purchased multiple tickets, children sometimes paid different fares, and different passenger groups may have traveled under different fare structures.

**List all available features:**

The dataset includes features such as survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, embark_town, alive, alone, family_size, and sex_num.

**Which other features could improve predictions and why?**

Passenger class (`pclass`) would likely improve predictions because first-class passengers generally paid higher fares than passengers traveling in lower classes.

**How many variables are in your Case 4?**

Three variables.

**Which variables did you choose for Case 4 and why do you feel those could make good inputs?**

I selected age, family size, and sex because together they provide more information about each passenger than a single feature alone. Combining these features improves the model's ability to predict fare.

## Section 4: Train a Regression Model (Linear Regression)

In [7]:
# Split each case into training and testing sets

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=123
)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=123
)

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=123
)

X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4, y4, test_size=0.2, random_state=123
)

In [8]:
# Train the regression models

lr_model1 = LinearRegression().fit(X1_train, y1_train)
lr_model2 = LinearRegression().fit(X2_train, y2_train)
lr_model3 = LinearRegression().fit(X3_train, y3_train)
lr_model4 = LinearRegression().fit(X4_train, y4_train)

In [9]:
# Predictions

y1_pred_train = lr_model1.predict(X1_train)
y1_pred_test = lr_model1.predict(X1_test)

y2_pred_train = lr_model2.predict(X2_train)
y2_pred_test = lr_model2.predict(X2_test)

y3_pred_train = lr_model3.predict(X3_train)
y3_pred_test = lr_model3.predict(X3_test)

y4_pred_train = lr_model4.predict(X4_train)
y4_pred_test = lr_model4.predict(X4_test)

In [10]:
def report(name, y_train, y_train_pred, y_test, y_test_pred):
    print("=" * 50)
    print(name)

    print("Training")
    print("R²:", round(r2_score(y_train, y_train_pred), 3))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_train, y_train_pred)), 2))
    print("MAE:", round(mean_absolute_error(y_train, y_train_pred), 2))

    print()

    print("Testing")
    print("R²:", round(r2_score(y_test, y_test_pred), 3))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 2))
    print("MAE:", round(mean_absolute_error(y_test, y_test_pred), 2))
    print()

In [11]:
report("Case 1 - Age", y1_train, y1_pred_train, y1_test, y1_pred_test)

report("Case 2 - Family Size", y2_train, y2_pred_train, y2_test, y2_pred_test)

report("Case 3 - Age + Family Size", y3_train, y3_pred_train, y3_test, y3_pred_test)

report(
    "Case 4 - Age + Family Size + Sex",
    y4_train,
    y4_pred_train,
    y4_test,
    y4_pred_test,
)

Case 1 - Age
Training
R²: 0.01
RMSE: 51.92
MAE: 28.89

Testing
R²: 0.003
RMSE: 37.97
MAE: 25.29

Case 2 - Family Size
Training
R²: 0.05
RMSE: 50.86
MAE: 27.8

Testing
R²: 0.022
RMSE: 37.61
MAE: 25.03

Case 3 - Age + Family Size
Training
R²: 0.073
RMSE: 50.23
MAE: 26.61

Testing
R²: 0.05
RMSE: 37.08
MAE: 24.28

Case 4 - Age + Family Size + Sex
Training
R²: 0.088
RMSE: 49.83
MAE: 26.0

Testing
R²: 0.119
RMSE: 35.7
MAE: 23.6



## Section 4 Reflection

### Compare the Training and Test Results

The training and testing results were generally similar across all four cases, indicating that the models did not significantly overfit the data. Models using multiple input features produced better prediction accuracy than models using a single feature.

### Did Case 1 overfit or underfit?

Case 1 underfit the data because age alone did not provide enough information to accurately predict passenger fare. Both the training and testing R² values were relatively low.

### Did Case 2 overfit or underfit?

Case 2 also underfit the data because family size alone was not a strong predictor of fare.

### Did Case 3 overfit or underfit?

Case 3 performed better than Cases 1 and 2 because it combined age and family size. However, it still underfit the data because prediction accuracy remained relatively low.

### Did Case 4 overfit or underfit?

Case 4 produced the best performance by combining age, family size, and sex. Although it outperformed the other cases, the model still underfit because additional variables such as passenger class would likely improve predictions.

### Did adding age improve the model?

Adding age improved prediction accuracy when combined with other features, but age alone was not a strong predictor of fare.

### Possible Explanation

Passenger fare depends much more on variables such as passenger class, cabin location, and ticket type than on age alone. This explains why adding age only modestly improved the model.

## Section 5: Compare Alternative Models

In [12]:
# Ridge Regression

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X4_train, y4_train)

y_pred_ridge = ridge_model.predict(X4_test)

print("Ridge Regression")
print("R²:", round(r2_score(y4_test, y_pred_ridge), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y4_test, y_pred_ridge)), 2))
print("MAE:", round(mean_absolute_error(y4_test, y_pred_ridge), 2))

Ridge Regression
R²: 0.119
RMSE: 35.71
MAE: 23.6


In [13]:
# Elastic Net

elastic_model = ElasticNet(alpha=0.3, l1_ratio=0.5)

elastic_model.fit(X4_train, y4_train)

y_pred_elastic = elastic_model.predict(X4_test)

print("Elastic Net")
print("R²:", round(r2_score(y4_test, y_pred_elastic), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y4_test, y_pred_elastic)), 2))
print("MAE:", round(mean_absolute_error(y4_test, y_pred_elastic), 2))

Elastic Net
R²: 0.101
RMSE: 36.06
MAE: 23.8


In [14]:
# Polynomial Regression

poly = PolynomialFeatures(degree=3)

X_train_poly = poly.fit_transform(X1_train)
X_test_poly = poly.transform(X1_test)

poly_model = LinearRegression()

poly_model.fit(X_train_poly, y1_train)

y_pred_poly = poly_model.predict(X_test_poly)

print("Polynomial Regression")
print("R²:", round(r2_score(y1_test, y_pred_poly), 3))
print("RMSE:", round(np.sqrt(mean_squared_error(y1_test, y_pred_poly)), 2))
print("MAE:", round(mean_absolute_error(y1_test, y_pred_poly), 2))

Polynomial Regression
R²: -0.003
RMSE: 38.1
MAE: 25.3


In [15]:
def compare_model(name, y_true, y_pred):
    print(f"{name}")
    print(f"R²   : {r2_score(y_true, y_pred):.3f}")
    print(f"RMSE : {np.sqrt(mean_squared_error(y_true, y_pred)):.2f}")
    print(f"MAE  : {mean_absolute_error(y_true, y_pred):.2f}")
    print()


compare_model("Linear Regression", y4_test, y4_pred_test)
compare_model("Ridge Regression", y4_test, y_pred_ridge)
compare_model("Elastic Net", y4_test, y_pred_elastic)
compare_model("Polynomial Regression", y1_test, y_pred_poly)

Linear Regression
R²   : 0.119
RMSE : 35.70
MAE  : 23.60

Ridge Regression
R²   : 0.119
RMSE : 35.71
MAE  : 23.60

Elastic Net
R²   : 0.101
RMSE : 36.06
MAE  : 23.80

Polynomial Regression
R²   : -0.003
RMSE : 38.10
MAE  : 25.30



## Section 5 Reflection

### What patterns does the cubic model capture?

The polynomial regression model captures nonlinear relationships between age and fare that are not represented by a simple straight line.

### Where does it perform well or poorly?

The model performs reasonably well for common age ranges but becomes less reliable near the extreme ends where fewer observations are available.

### Did polynomial regression outperform linear regression?

The polynomial model produced a more flexible fit, but it did not substantially outperform the best linear regression model because age alone is not a strong predictor of fare.

### Which model performed best?

The Linear Regression model using **Age, Family Size, and Sex (Case 4)** produced the best overall performance.

### Did regularization improve the model?

The Ridge Regression and Elastic Net models produced similar performance while reducing the possibility of overfitting.

### Did the regularized model reduce overfitting?

Yes. Regularization helps control model complexity by shrinking large coefficients and improving generalization.

## Section 6: Final Thoughts & Insights

### Summary of Findings

This project compared several regression models for predicting Titanic passenger fares. Four different combinations of input features were evaluated using Linear Regression, followed by Ridge Regression, Elastic Net, and Polynomial Regression.

Among the linear regression models, **Case 4**, which combined age, family size, and sex, produced the strongest performance.

The regularized models produced similar results while helping reduce overfitting. Polynomial regression captured nonlinear patterns but did not significantly improve prediction accuracy because age alone was not a strong predictor.

### Challenges

The most difficult part of this project was selecting features that meaningfully predicted fare. Many variables had only a limited relationship with ticket price.

### Future Improvements

Future work could include adding passenger class (`pclass`), embarkation port, cabin information, or additional engineered features to improve prediction accuracy.

### Final Reflection

This project strengthened my understanding of regression analysis, feature selection, regularization, and model evaluation using MAE, RMSE, and R². These techniques can be applied to many real-world prediction problems, including housing prices, insurance costs, and sales forecasting.